In [0]:
# ============================================================
# 03A_dim_prepare.ipynb
# Mục tiêu:
# - Build distinct content candidates from most_search_t6/t7
# - Apply exact alias rules
# - Split into dim_content_seed and dim_content_unresolved
# - Export unresolved + rules for local/Colab labeling
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.types import StringType
import unicodedata
import re

In [0]:
# CONFIG
CATALOG = "workspace"
SCHEMA = "customer360"
DB = f"{CATALOG}.{SCHEMA}"

TABLE_MOST_SEARCH_T6 = f"{DB}.new_most_search_t6"
TABLE_MOST_SEARCH_T7 = f"{DB}.new_most_search_t7"
TABLE_RULES = f"{DB}.dim_content_new_rules"

TABLE_DIM_SEED = f"{DB}.dim_content_new_seed"
TABLE_DIM_UNRESOLVED = f"{DB}.dim_content_new_unresolved"

# Export location - use Unity Catalog Volume
EXPORT_BASE = "/Volumes/workspace/customer360/c360_volume"
PATH_EXPORT_UNRESOLVED = f"{EXPORT_BASE}/dim_content_new_unresolved"
PATH_EXPORT_RULES = f"{EXPORT_BASE}/dim_content_new_rules"

In [0]:
# Normalize text function
def normalize_text_py(text: str) -> str:
    if text is None:
        return None

    text = str(text).strip().lower()
    if not text:
        return None

    text = re.sub(r"[\"'`]+", "", text)
    text = re.sub(r"[_\-]+", " ", text)
    text = re.sub(r"[^\w\s]", " ", text, flags=re.UNICODE)
    text = re.sub(r"\s+", " ", text).strip()

    if not text:
        return None

    text_no_accent = unicodedata.normalize("NFD", text)
    text_no_accent = "".join(
        ch for ch in text_no_accent if unicodedata.category(ch) != "Mn"
    )
    text_no_accent = re.sub(r"\s+", " ", text_no_accent).strip()

    return text_no_accent if text_no_accent else None

normalize_text_udf = F.udf(normalize_text_py, StringType())

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/udf.py:103: UserWarning: Cannot infer the eval type from type hints. 
  warnings.warn("Cannot infer the eval type from type hints. ", UserWarning)


In [0]:
# LOAD MOST SEARCH TABLES
most_search_t6 = spark.table(TABLE_MOST_SEARCH_T6)
most_search_t7 = spark.table(TABLE_MOST_SEARCH_T7)

In [0]:
# LOAD RULES TABLES
rules_df = (
    spark.table(TABLE_RULES)
    .select("alias", "canonical_content", "category")
    .filter(F.col("alias").isNotNull())
    .filter(F.col("canonical_content").isNotNull())
    .withColumn("alias_norm", normalize_text_udf(F.col("alias")))
    .filter(F.col("alias_norm").isNotNull())
    .select("alias_norm", "canonical_content", "category")
    .dropDuplicates(["alias_norm"])
)

In [0]:
# BUILD DISTINCT CONTENT CANDIDATES
dim_candidates = (
    most_search_t6
    .select(F.col("Most_Search").alias("content"))
    .union(
        most_search_t7.select(F.col("Most_Search").alias("content"))
    )
    .distinct()
    .filter(F.col("content").isNotNull())
    .withColumn("content_norm", normalize_text_udf(F.col("content")))
    .filter(F.col("content_norm").isNotNull())
)

In [0]:
# APPLY RULES
dim_prepared = (
    dim_candidates.alias("c")
    .join(
        rules_df.alias("r"),
        F.col("c.content_norm") == F.col("r.alias_norm"),
        "left"
    )
    .withColumn(
        "canonical_content",
        F.coalesce(F.col("r.canonical_content"), F.col("c.content"))
    )
    .select(
        F.col("c.content").alias("content"),
        F.col("canonical_content"),
        F.col("r.category").alias("category")
    )
)

In [0]:
# SPLIT INTO SEED / UNRESOLVED
dim_content_seed = (
    dim_prepared
    .filter(F.col("category").isNotNull())
    .dropDuplicates(["content"])
)

dim_content_unresolved = (
    dim_prepared
    .filter(F.col("category").isNull())
    .select("content", "canonical_content")
    .dropDuplicates(["content"])
)

In [0]:
# SAVE TABLES
dim_content_seed.write.mode("overwrite").saveAsTable(TABLE_DIM_SEED)
dim_content_unresolved.write.mode("overwrite").saveAsTable(TABLE_DIM_UNRESOLVED)

In [0]:
# EXPORT FOR LOCAL/COLAB LABELING
# unresolved
dim_content_unresolved.write.mode("overwrite").parquet(PATH_EXPORT_UNRESOLVED)

# export original rules table (not normalized version)
spark.table(TABLE_RULES).write.mode("overwrite").parquet(PATH_EXPORT_RULES)


In [0]:
# Checking results
print("dim_content_seed:", spark.table(TABLE_DIM_SEED).count())
print("dim_content_unresolved:", spark.table(TABLE_DIM_UNRESOLVED).count())

print("\nSample dim_content_seed:")
spark.table(TABLE_DIM_SEED).show(20, truncate=False)

print("\nSample dim_content_unresolved:")
spark.table(TABLE_DIM_UNRESOLVED).show(20, truncate=False)

print("\nExport completed:")
print("-", PATH_EXPORT_UNRESOLVED)
print("-", PATH_EXPORT_RULES)

dim_content_seed: 7
dim_content_unresolved: 693

Sample dim_content_seed:
+--------------------+--------------------+------------+
|content             |canonical_content   |category    |
+--------------------+--------------------+------------+
|vtv                 |vtv                 |News        |
|penthouse           |penthouse           |K-Drama     |
|bolero              |bolero              |Music       |
|Việt Nam            |Việt Nam            |Sport       |
|taxi driver         |taxi driver         |Action      |
|running man         |running man         |Reality show|
|thử thách thần tượng|thử thách thần tượng|Reality show|
+--------------------+--------------------+------------+


Sample dim_content_unresolved:
+------------------------------------+------------------------------------+
|content                             |canonical_content                   |
+------------------------------------+------------------------------------+
|giao hữu bóng đá việt nam           |